> **AI-generated.** An AI assistant drafted this page, and no maintainer has reviewed it yet. Please report errors on the issue tracker.

# Tutorial 3: Revising and cross-checking the model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TARPS-group/prob-pipe/blob/dev/overhaul/docs/tutorials/03_revising.ipynb)

[Tutorial 2](02_fitting.ipynb) found that the moose counts jump from year to year more than the model allows.
This tutorial adds that variation to the model as a new stage and fits the revised model.
It then asks whether the posterior is right, by fitting the model again with other algorithms and by writing it in PyMC.

It develops four ideas of [how ProbPipe works](../index.md#how-it-works):

1. **Mathematical objects:** a model is composed of stages, each a distribution or a conditional distribution.
2. **One vocabulary of operations:** one `condition_on` call fits the model with any algorithm and in either language.
3. **Computation from capabilities:** `condition_on` selects an algorithm, and `with_options` chooses another one, sets its settings, or requires an exact answer.
4. **Native Python and existing packages:** a model written in PyMC is a ProbPipe object like the native one.

The tutorial takes about 25 minutes, and it ends with three exercises.

In [1]:
# On Google Colab, install ProbPipe with PyMC and nutpie, and download the data; elsewhere this cell does nothing.
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import urllib.request

    %pip install --quiet "probpipe-core[pymc,nutpie] @ git+https://github.com/TARPS-group/prob-pipe.git@dev/overhaul"
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/TARPS-group/prob-pipe/dev/overhaul/"
        "docs/tutorials/data/moose.csv",
        "data/moose.csv",
    )

In [2]:
import warnings

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

from probpipe import (
    Beta,
    HalfNormal,
    LogNormal,
    Normal,
    Poisson,
    PyMCModel,
    ResolutionError,
    condition_on,
    conditional_distribution,
    mean,
    predictive_check,
    workflow_run,
)
from probpipe.diagnostics import add_mcmc_diagnostics

# TensorFlow Probability warns that JAX runs in 32-bit precision, which these models use.
warnings.filterwarnings("ignore", message=r"Explicitly requested dtype float64")
# PyTensor warns that it cannot fuse the loop of section 7's PyMC model, which affects only its speed.
warnings.filterwarnings("ignore", message=r"Loop fusion failed")

data = pd.read_csv("data/moose.csv")
years = data["year"].to_numpy()
counts = jnp.asarray(data["count"], dtype=jnp.float32)
YEARS = counts.shape[0]
observed = {"y": counts}

## 1. Process noise as a stage

A population's growth varies from year to year with the weather, food, and predators.
We add a **shock** $\epsilon_t$ to each year's growth:

$$N_{t+1} = N_t \exp\left(r\left(1 - \frac{N_t}{K}\right) + \epsilon_t\right), \qquad \epsilon_t \sim \text{Normal}(0, \sigma).$$

The scale $\sigma$ of the shocks is a new parameter, whose prior expects shocks of a few percent of the growth per year.
`trajectory` gains an argument for the shocks, and with shocks of zero it is tutorial 1's recursion.

In [3]:
def trajectory(r: jax.Array, K: jax.Array, n0: float, shocks: jax.Array) -> jax.Array:
    """The population in each year after the year of *n0*, given each year's shock to the growth."""

    def step(N: jax.Array, shock: jax.Array) -> tuple[jax.Array, jax.Array]:
        N_next = N * jnp.exp(r * (1.0 - N / K) + shock)
        return N_next, N_next

    _, N = jax.lax.scan(step, jnp.asarray(n0, dtype=jnp.float32), shocks)
    return N


prior = (
    LogNormal("K", jnp.log(350.0), 0.4)
    * LogNormal("r", jnp.log(0.4), 0.4)
    * Beta("phi", 2.0, 2.0)
    * HalfNormal("sigma", 0.2)
).with_label("prior")
print("components of the prior:", list(prior.event_spec.components))

components of the prior: ['K', 'r', 'phi', 'sigma']


The shocks depend on their scale, so their distribution given $\sigma$ is a conditional distribution, a stage of the model between the prior and the counts.
The counts now depend on the shocks as well as on $K$, $r$, and $\phi$.

In [4]:
def shocks_given_sigma(sigma: jax.Array) -> Normal:
    return Normal("shocks", jnp.zeros(YEARS), sigma)


shocks = conditional_distribution(
    "shocks", shocks_given_sigma, given_spec={"sigma": prior.event_spec.components["sigma"]}
)


def noisy_counts(
    K: jax.Array, r: jax.Array, phi: jax.Array, shocks: jax.Array, n0: float = 50.0
) -> Poisson:
    return Poisson("y", phi * trajectory(r, K, n0, shocks))


inputs = {name: prior.event_spec.components[name] for name in ["K", "r", "phi"]}
likelihood = conditional_distribution(
    "counts", noisy_counts, given_spec={**inputs, "shocks": shocks.event_spec.components["shocks"]}
)
model = (likelihood * shocks * prior).with_label("model")
print("components of the model:", list(model.event_spec.components))

components of the model: ['y', 'shocks', 'K', 'r', 'phi', 'sigma']


`*` composed three objects: the counts given the parameters and the shocks, the shocks given their scale, and the prior.
Each stage is a distribution or a conditional distribution, and `*` matches each stage's inputs to the components of the stages after it.

## 2. The revised fit

`condition_on` fits the revised model as it fit tutorial 2's:

In [5]:
with workflow_run(seed=4):
    posterior = condition_on(model, observed)
add_mcmc_diagnostics(posterior)
diagnostics = posterior.diagnostics.mcmc
print("method:", posterior.provenance.metadata["method"])
print("number of parameters, counting each shock:", len(diagnostics.rhat))
print("largest R-hat:", round(max(diagnostics.rhat.values()), 3))
print("smallest bulk ESS:", round(min(diagnostics.ess_bulk.values())))
print("divergent transitions:", diagnostics.n_divergences)

method: blackjax_nuts
number of parameters, counting each shock: 26
largest R-hat: 1.003
smallest bulk ESS: 1214
divergent transitions: 0


The chains have mixed for all 26 parameters: $K$, $r$, $\phi$, $\sigma$, and the 22 shocks.
Tutorial 2's predictive check now asks whether the revised model jumps from year to year as the counts do:

In [6]:
def change_spread(y: jax.Array) -> jax.Array:
    return jnp.std(jnp.diff(y))


def largest_count(y: jax.Array) -> jax.Array:
    return jnp.max(y)


with workflow_run(seed=5):
    check = predictive_check(likelihood, posterior, [change_spread, largest_count], counts)
for name in ["change_spread", "largest_count"]:
    print(
        f"{name}: observed {float(check[f'{name}/observed_statistic']):.1f},"
        f" p-value {float(check[f'{name}/p_value']):.3f}"
    )

change_spread: observed 31.3, p-value 0.374
largest_count: observed 296.0, p-value 0.510


Neither statistic now separates the observed counts from the replications, where the spread of the changes had a p-value of 0.004 in tutorial 2.
The replications keep each draw's shocks, so the check tests how the counts scatter around the population that the posterior infers.

## 3. A second algorithm

Is the posterior right?
One check fits the same model with a different algorithm.
`with_options(method=...)` chooses the inference method, here TensorFlow Probability's implementation of NUTS in place of BlackJAX's:

In [7]:
with workflow_run(seed=6):
    tfp_posterior = condition_on.with_options(method="tfp_nuts")(model, observed)
add_mcmc_diagnostics(tfp_posterior)
print("method:", tfp_posterior.provenance.metadata["method"])
print("divergent transitions:", tfp_posterior.diagnostics.mcmc.n_divergences)

parameters = ["K", "r", "phi", "sigma"]
pd.DataFrame(
    {
        "BlackJAX NUTS": [float(mean(posterior[name])) for name in parameters],
        "TFP NUTS": [float(mean(tfp_posterior[name])) for name in parameters],
    },
    index=[f"mean of {name}" for name in parameters],
).round(3)

method: tfp_nuts
divergent transitions: 22


,BlackJAX NUTS,TFP NUTS
mean of K,438.541,442.690
mean of r,0.281,0.283
mean of phi,0.594,0.591
mean of sigma,0.125,0.127


The call and the kind of result are the same, and the posterior means agree.
TFP's sampler reports divergent transitions, where BlackJAX's reported none, so its draws need a closer look.

## 4. A setting of the algorithm

A method's settings are its `method_options`.
During warmup, NUTS adapts its step size until it accepts a target share of its proposals, and TFP's default target of 0.75 is lower than BlackJAX's 0.8.
A higher target adapts a smaller step, which follows the posterior more closely where it curves sharply.
TFP's sampler also mixes more slowly than BlackJAX's, which adapts to the scale of each parameter during warmup, so we give it 2,000 draws per chain:

In [8]:
with workflow_run(seed=6):
    careful = condition_on.with_options(
        method="tfp_nuts", method_options={"target_accept_prob": 0.9, "num_results": 2000}
    )(model, observed)
add_mcmc_diagnostics(careful)
print("divergent transitions:", careful.diagnostics.mcmc.n_divergences)
print("largest R-hat:", round(max(careful.diagnostics.mcmc.rhat.values()), 3))
print("mean of K:", round(float(mean(careful["K"])), 1))

divergent transitions: 0
largest R-hat: 1.006
mean of K: 444.4


With these settings, TFP's sampler reports no divergences, its R-hat is below 1.01, and its posterior matches BlackJAX's.

## 5. An algorithm that struggles

Not every algorithm suits every posterior.
Random-walk Metropolis proposes each move at random, without the gradient that NUTS follows, and on 26 parameters it moves slowly:

In [9]:
with workflow_run(seed=7):
    random_walk = condition_on.with_options(method="blackjax_rwmh")(model, observed)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    add_mcmc_diagnostics(random_walk)
walk = random_walk.diagnostics.mcmc
print("largest R-hat:", round(max(walk.rhat.values()), 2))
print("smallest bulk ESS:", round(min(walk.ess_bulk.values())))

largest R-hat: 3.09
smallest bulk ESS: 5


The chains disagree, and 4,000 draws are worth only a handful of independent ones, so these draws do not describe the posterior.
The diagnostics, which apply to every sampler's result, make the failure visible.

## 6. Asking for an exact answer

`exact_only=True` refuses every approximate method.
No exact method computes this posterior, so `condition_on` raises an error that lists each route it tried and why it declined:

In [10]:
try:
    condition_on.with_options(exact_only=True)(model, observed)
except ResolutionError as error:
    summary, _, attempts = str(error).partition(" Tried: ")
    print("error:", summary)
    for attempt in attempts.split("; ")[:4]:
        print("tried", attempt)

error: condition_on: no route applies with exact_only.
tried curry (exact methods): route 'curry' declined: the conditioned object is not a kernel
tried slice (exact methods): route 'slice' declined: the factor producing ['y'] conditions on ['K', 'phi', 'r', 'shocks'], which the given leaves free, so the conditional is not a product of the factors
tried exact_conditioning: FactoredDistribution does not claim SupportsExactConditioning
tried inference_methods (exact methods): the exact stage's result is unnormalized, and no exact method normalizes it


## 7. The same model in PyMC

Many models are already written in a probabilistic programming language such as PyMC or Stan.
A PyMC program is a ProbPipe object too: `PyMCModel` wraps a function that builds the program, and its components are the program's variables.
We write the revised model in PyMC, with the same priors and the same recursion:

In [11]:
import pymc as pm
import pytensor.tensor as pt


def build_pymc_model(y: np.ndarray | None = None) -> pm.Model:
    with pm.Model() as pymc_model:
        K = pm.LogNormal("K", mu=np.log(350.0), sigma=0.4)
        r = pm.LogNormal("r", mu=np.log(0.4), sigma=0.4)
        phi = pm.Beta("phi", alpha=2.0, beta=2.0)
        sigma = pm.HalfNormal("sigma", sigma=0.2)
        shocks = pm.Normal("shocks", mu=0.0, sigma=sigma, shape=YEARS)
        N = 50.0
        expected = []
        for year in range(YEARS):
            N = N * pt.exp(r * (1.0 - N / K) + shocks[year])
            expected.append(phi * N)
        pm.Poisson("y", mu=pt.stack(expected), observed=y)
    return pymc_model


model_in_pymc = PyMCModel("model in PyMC", build_pymc_model)
print("components of the PyMC model:", list(model_in_pymc.event_spec.components))

report = condition_on.check(model_in_pymc, observed)
print("route:", report.route)
print("method:", report.method)

components of the PyMC model: ['K', 'r', 'phi', 'sigma', 'shocks', 'y']


route: inference_methods
method: nutpie_nuts


The same `condition_on` call conditions it on the counts.
It selects nutpie's NUTS, which samples PyMC programs fastest; nutpie compiles the program first, which takes a minute or two.
Its setting `progress_bar=False` keeps it from drawing a progress bar.

In [12]:
with workflow_run(seed=8):
    pymc_posterior = condition_on.with_options(method_options={"progress_bar": False})(
        model_in_pymc, {"y": np.asarray(counts)}
    )
add_mcmc_diagnostics(pymc_posterior)
print("largest R-hat:", round(max(pymc_posterior.diagnostics.mcmc.rhat.values()), 3))

pd.DataFrame(
    {
        "ProbPipe model": [float(mean(posterior[name])) for name in parameters],
        "PyMC model": [float(mean(pymc_posterior[name])) for name in parameters],
    },
    index=[f"mean of {name}" for name in parameters],
).round(3)

largest R-hat: 1.007


,ProbPipe model,PyMC model
mean of K,438.541,441.251
mean of r,0.281,0.281
mean of phi,0.594,0.594
mean of sigma,0.125,0.125


The two programs of one model give one posterior.
The PyMC posterior is a distribution like the others, so every operation of these tutorials applies to it, and the next tutorials could use it in place of the native one.

## Exercises

Each exercise is followed by a solution, which Colab shows collapsed.

**Exercise 1.** Fit the PyMC model with PyMC's own NUTS, `method="pymc_nuts"`, and compare its posterior means with nutpie's.

In [13]:
# @title Solution
with workflow_run(seed=9):
    pymc_nuts_posterior = condition_on.with_options(method="pymc_nuts")(
        model_in_pymc, {"y": np.asarray(counts)}
    )
for name in parameters:
    print(
        f"mean of {name}: {float(mean(pymc_nuts_posterior[name])):.3f} with pymc_nuts,"
        f" {float(mean(pymc_posterior[name])):.3f} with nutpie"
    )

Output()

mean of K: 439.410 with pymc_nuts, 441.251 with nutpie
mean of r: 0.285 with pymc_nuts, 0.281 with nutpie
mean of phi: 0.590 with pymc_nuts, 0.594 with nutpie
mean of sigma: 0.126 with pymc_nuts, 0.125 with nutpie


**Exercise 2.** Shocks may come from a distribution with heavier tails than the normal, as a severe winter would.
Replace the shocks' stage with `StudentT("shocks", 4.0, jnp.zeros(YEARS), sigma)`, compose the model again, and compare the posterior of $\sigma$.

In [14]:
# @title Solution
from probpipe import StudentT

def heavy_shocks_given_sigma(sigma: jax.Array) -> StudentT:
    return StudentT("shocks", 4.0, jnp.zeros(YEARS), sigma)


heavy_shocks = conditional_distribution(
    "shocks", heavy_shocks_given_sigma, given_spec={"sigma": prior.event_spec.components["sigma"]}
)
heavy_model = (likelihood * heavy_shocks * prior).with_label("model with heavy-tailed shocks")
with workflow_run(seed=10):
    heavy_posterior = condition_on(heavy_model, observed)
add_mcmc_diagnostics(heavy_posterior)
print("divergent transitions:", heavy_posterior.diagnostics.mcmc.n_divergences)
print(f"mean of sigma, normal shocks: {float(mean(posterior['sigma'])):.3f}")
print(f"mean of sigma, Student-t shocks: {float(mean(heavy_posterior['sigma'])):.3f}")
# A Student-t distribution with scale sigma has standard deviation sigma * sqrt(2) for 4
# degrees of freedom, so a smaller sigma describes shocks of a similar size.

divergent transitions: 0
mean of sigma, normal shocks: 0.125
mean of sigma, Student-t shocks: 0.100


**Exercise 3.** Fit the revised model with BlackJAX's Hamiltonian Monte Carlo, `method="blackjax_hmc"`, which takes a fixed number of steps per move.
Do its diagnostics show that it mixed?

In [15]:
# @title Solution
with workflow_run(seed=11):
    hmc_posterior = condition_on.with_options(method="blackjax_hmc")(model, observed)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    add_mcmc_diagnostics(hmc_posterior)
hmc = hmc_posterior.diagnostics.mcmc
print("largest R-hat:", round(max(hmc.rhat.values()), 3))
print("smallest bulk ESS:", round(min(hmc.ess_bulk.values())))
print("mean of K:", round(float(mean(hmc_posterior["K"])), 1))
# An R-hat above 1.01 and an effective sample size near 100 say that it has not mixed
# well: a fixed number of steps per move suits this posterior less well than NUTS.

largest R-hat: 1.049
smallest bulk ESS: 107
mean of K: 444.9


## Summary

- A model is composed of stages: here the prior, the shocks given their scale, and the counts given the parameters and the shocks.
- With process noise, the model passes the predictive check that tutorial 2's model failed.
- `with_options(method=...)` fits the same model with another algorithm, `method_options` sets the algorithm's settings, and `exact_only=True` refuses approximate methods. The diagnostics tell whether an algorithm's draws can be trusted.
- A model written in PyMC is a ProbPipe object, which the same `condition_on` call fits, and its posterior agrees with the native model's.

[Tutorial 4](04_forecasting.ipynb) forecasts the population with the revised model.